# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sujithauday/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Among pages that already get meaningful traffic and already show CTR below what their position tier normally earns, predict which ones will keep declining — those go to the top of the SEO team’s queue.
This is a CTR/Engagement Opportunity Scoring lane.
The ML task is ranking/scoring, producing an opportunity score and a top‑50 priority list for SEO.

One row = one web page(content_id)

In [7]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs")
con.execute(f"""
    CREATE SECRET hf_token_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

rel = "hf://datasets/FlyRank/internship-warehouse"
table = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"
con.sql(f"""
    SELECT COUNT(*)
    FROM {table}
""").show()
# Column names

schema = con.sql(f"DESCRIBE SELECT * FROM {table}").df()
print(schema["column_name"].tolist())
top3 = con.sql(f"SELECT * FROM {table} LIMIT 3").df()
print(top3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
  report_date           client_hash_id           content_hash_id  \
0  2025-01-27  client_9958f0a7ae1df715  content_3b70a18ea133b2bb   
1  2025-01-27  client_9958f0a7ae1df715  content_fe8e8155ce1d47a2   
2  2025-01-27  client_9958f0a7ae1df715  content_b4462a1b90640058   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0           

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

FACT_DAILY = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"
summary = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date
    FROM {FACT_DAILY}
""").df()
# print(summary.columns.tolist())
print(summary)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  clients  content_items first_report_date last_report_date
0    78835655       70         427292        2025-01-27       2026-06-30


In [8]:
MONTH = '2025-02'
table2 = f"""
    read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
"""
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS duplicate_count
    FROM {table2}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f'Duplicate raw-grain rows found in month={MONTH}: {len(grain_check)}')
if len(grain_check) == 0:
    print('Result: zero rows — the grain holds (report_date × client_hash_id × content_hash_id).')
else:
    display(grain_check)

Duplicate raw-grain rows found in month=2025-02: 0
Result: zero rows — the grain holds (report_date × client_hash_id × content_hash_id).


In [18]:
# get data for 4 months:
MONTHS = ['2026-01', '2026-03', '2025-12', '2026-02']
paths = ", ".join([f"'{rel}/fact_content_daily_performance/month={m}/*.parquet'" for m in MONTHS])

coverage = con.sql(f"""
    SELECT
        client_hash_id,
        MIN(report_date) AS first_seen,
        MAX(report_date) AS last_seen,
        COUNT(DISTINCT report_date) AS days_present
    FROM read_parquet([{paths}])
    GROUP BY client_hash_id
    ORDER BY first_seen
""").df()

print(f"Clients with any data in Dec 2025 - Mar 2026: {len(coverage)} of 70 total")
print(coverage.sort_values(by = "days_present", ascending = True))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clients with any data in Dec 2025 - Mar 2026: 59 of 70 total
             client_hash_id first_seen  last_seen  days_present
58  client_e00b29e582949543 2026-03-23 2026-03-31             9
57  client_810019792c9b8efc 2026-03-20 2026-03-31            12
56  client_f6f0cdf26d03d7bd 2026-03-19 2026-03-31            13
55  client_86ebc2f12c01f586 2026-03-03 2026-03-31            29
54  client_b77d0d5f08f05e64 2026-03-01 2026-03-31            31
53  client_7eafe750768f0ac2 2026-02-26 2026-03-31            34
46  client_157ffe4d4a595515 2026-02-19 2026-03-31            41
47  client_20259bd6705d81d4 2026-02-19 2026-03-31            41
52  client_e5c2aa26a8598242 2026-02-19 2026-03-31            41
49  client_2b4306c3ed003f01 2026-02-19 2026-03-31            41
51  client_a80fca3f171ed1de 2026-02-19 2026-03-31            41
50  client_3f0ce4d44fe94f3d 2026-02-19 2026-03-31            41
48  client_1a730cb2640a1abf 2026-02-19 2026-03-31            41
45  client_0fa64a184f18a4a0 2026-02-17 2026

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.